In [2]:
import os
import csv
import scanpy as sc

# Inställningar
input_path = "/data/shared/alzgene26/PathwaysLA/binn_connectivity.csv"
output_path = "/data/shared/alzgene26/PathwaysLA/binn_cutted3.csv"
file_temp = "/data/shared/alzgene26/data/conv_data/OPCs.h5ad"

# 1. Ladda initiala gener och rensa eventuella mellanslag
if os.path.exists(file_temp):
    big = sc.read_h5ad(file_temp, backed="r")
    # Vi lägger till .strip() för säkerhets skull
    initial_genes = set(str(g).strip() for g in big.var_names[:30])
else:
    print("Filen hittades inte")
    initial_genes = set()
    
# 2. Bygg adj_list med .strip() på alla noder
adj_list = {}
with open(input_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) < 2: continue
        source, target = row[0].strip(), row[1].strip()
        if source not in adj_list:
            adj_list[source] = []
        adj_list[source].append(target)

In [3]:
print(adj_list["ADAM10"])

['R-HSA-2122948', 'R-HSA-977225', 'R-HSA-422475', 'R-HSA-1442490', 'R-HSA-2691232', 'R-HSA-2894862', 'R-HSA-2644606', 'R-HSA-2660826', 'R-HSA-1474228', 'R-HSA-1266738', 'R-HSA-1643685', 'R-HSA-5663202', 'R-HSA-2682334', 'R-HSA-3928665', 'R-HSA-1474244', 'R-HSA-168256', 'R-HSA-168249', 'R-HSA-392499', 'R-HSA-2979096', 'R-HSA-9013507', 'R-HSA-9013700', 'R-HSA-9675108', 'R-HSA-6798695', 'R-HSA-597592', 'R-HSA-8957275', 'R-HSA-381426', 'R-HSA-162582', 'R-HSA-177929', 'R-HSA-157118', 'R-HSA-1980143', 'R-HSA-2691230', 'R-HSA-2894858', 'R-HSA-2644602', 'R-HSA-2644603', 'R-HSA-2660825', 'R-HSA-1980145', 'R-HSA-9012852', 'R-HSA-9013694', 'R-HSA-9006934']


In [3]:
initial_genes = {"R-HSA-1442490"}
output_path = "/data/shared/alzgene26/PathwaysLA/binn_cutted3.csv"

# Rensa målfilen
open(output_path, "w").close()

# Starta traversering
layer = initial_genes.copy()
infeasible = initial_genes.copy()

print(f"Startar med {len(layer)} gener...")

while layer:
    temp_layer = set()
    edges_to_write = []
    
    # Processa nuvarande lager
    while layer:
        current_node = layer.pop()
        if current_node in adj_list:
            for target_node in adj_list[current_node]:
                if target_node not in infeasible:
                    temp_layer.add(target_node)
                    edges_to_write.append((current_node, target_node))
    
    # Om vi hittade nya kanter, skriv ner dem
    if edges_to_write:
        with open(output_path, "a", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(edges_to_write) # Mycket snabbare än writerow i loop
        
        print(f"Hittade {len(edges_to_write)} nya kopplingar. Nästa lager har {len(temp_layer)} noder.")
    
    # Uppdatera inför nästa varv
    infeasible.update(temp_layer)
    layer = temp_layer

Startar med 1 gener...
Hittade 1 nya kopplingar. Nästa lager har 1 noder.
Hittade 1 nya kopplingar. Nästa lager har 1 noder.
